In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_expenses_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #      "ExpensesId string,"
        #      "SourceSystem string,"
        #      "PortfolioKey string,"
        #      "PortfolioName string,"
        #      "IngestionTimestamp timestamp,"
        #      "BatchId string,"
        #      "ExpensesKey string,"
        #      "PartyKey string,"
        #      "CDI string,"
        #      "ExposureKey string,"
        #      "AgreementKey string,"
        #      "LegalActionKey string,"
        #      "InsuranceKey string,"
        #      "ExpensesTypeCode string,"
        #      "ExpensesType string,"
        #      "ExpensesDate timestamp,"
        #      "TotalAmount decimal(20,6),"
        #      "NetAmount decimal(20,6),"
        #      "TaxAmount decimal(19,4),"
        #      "VatAmount decimal(20,6),"
        #      "InternalCost decimal(19,10),"
        #      "OtherExpenses decimal(19,10),"
        #      "ExpensesCategoryCode string,"
        #      "ExpensesCategory string,"
        #      "ExpensesStatusCode string,"
        #      "ExpensesStatus string,"
        #      "VATPercentage string,"
        #      "PacketCode int,"
        #      "Packet string,"
        #      "Comments string,"
        #      "LawyerKey int,"
        #      "ExternalCompanyPartyKey int,"
        #      "InvoiceDate timestamp,"
        #      "InvoiceNumber string,"
        #      "InvoiceApprovalStatusCode int,"
        #      "InvoiceApprovalStatus int,"
        #      "ExpensesValidationStatusCode int,"
        #      "ExpensesValidationStatus int,"
        #      "ExpensesApprovalStatusCode int,"
        #      "ExpensesApprovalStatus int,"
        #      "BusinessDate date"
        #     )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Expenses"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ============================== fs_legal_expenses_ext ====================
        fs_legal_expenses_ext_extra_filters = None
        if fs_legal_expenses_ext_extra_filters is not None:
            fs_legal_expenses_ext_extra_filters.append(date_filter)
            fs_legal_expenses_ext_final_filters = " AND ".join(
                fs_legal_expenses_ext_extra_filters
            )
        else:
            fs_legal_expenses_ext_final_filters = date_filter
        fs_legal_expenses_ext_selected_fields = [
            "LEED_CODE",
            "LGET_ID",
            "FSLD_TOTAL_AMT_LCY",
            "PD_DESC",
            "CAC_CODE",
            "FSLD_NET_AMT_LCY",
            "FSLD_TAX_AMT_LCY",
            "FSLD_VAT_AMT_LCY",
            "EXTC_ID",
            "LEE_INVOICE_DATE",
            "LEE_INVOICE_NUMBER",
            "LEEA_ID",
            "LEEDV_ID",
            "LEEDA_ID",
            "LAWY_ID",
            "CASE_CODE",
            "LGET_ID",
            "Businessdate",
        ]

        fs_legal_expenses_ext = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_legal_expenses_ext",
            fs_legal_expenses_ext_final_filters,
            fs_legal_expenses_ext_selected_fields,
        )

        # ============================== fs_cases_snap ====================
        fs_cases_snap_extra_filters = None
        if fs_cases_snap_extra_filters is not None:
            fs_cases_snap_extra_filters.append(date_filter)
            fs_cases_snap_final_filters = " AND ".join(fs_cases_snap_extra_filters)
        else:
            fs_cases_snap_final_filters = date_filter
        fs_cases_snap_selected_fields = [
            "CUST_CODE",
            "case_code",
            "SPET_CODE",
        ]

        fs_cases_snap = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_cases_snap",
            fs_cases_snap_final_filters,
            fs_cases_snap_selected_fields,
        )

        # ============================== dim_cases_static ====================
        dim_cases_static_extra_filters = None
        if dim_cases_static_extra_filters is not None:
            dim_cases_static_extra_filters.append(date_filter)
            dim_cases_static_final_filters = " AND ".join(
                dim_cases_static_extra_filters
            )
        else:
            dim_cases_static_final_filters = date_filter

        dim_cases_static_selected_fields = [
            "case_code",
            "CASE_CONTRACT_ACCOUNT_KEY",
            "CASE_ACCOUNT_FOR_SEARCH",
        ]

        dim_cases_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_cases_static",
            dim_cases_static_final_filters,
            dim_cases_static_selected_fields,
        )

        # ============================== fs_activity_leaf ====================
        fs_activity_leaf_extra_filters = None
        if fs_activity_leaf_extra_filters is not None:
            fs_activity_leaf_extra_filters.append(date_filter)
            fs_activity_leaf_final_filters = " AND ".join(
                fs_activity_leaf_extra_filters
            )
        else:
            fs_activity_leaf_final_filters = date_filter

        fs_activity_leaf_selected_fields = [
            "FSAC_INTERNAL_COST_LCY",
            "FSAC_OTHER_EXPENSES_LCY",
            "PACK_ID",
            "CAC_CODE",
        ]

        fs_activity_leaf = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_activity_leaf",
            fs_activity_leaf_final_filters,
            fs_activity_leaf_selected_fields,
        )

        # ============================== dim_legal_expense_type ====================
        dim_legal_expense_type_extra_filters = None
        if dim_legal_expense_type_extra_filters is not None:
            dim_legal_expense_type_extra_filters.append(date_filter)
            dim_legal_expense_type_final_filters = " AND ".join(
                dim_legal_expense_type_extra_filters
            )
        else:
            dim_legal_expense_type_final_filters = date_filter

        dim_legal_expense_type_selected_fields = [
            "LGET_ID",
            "LGET_NAME",
        ]

        dim_legal_expense_type = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_legal_expense_type",
            dim_legal_expense_type_final_filters,
            dim_legal_expense_type_selected_fields,
        )

        # ============================== at_lawyers ====================
        at_lawyers_extra_filters = None
        if at_lawyers_extra_filters is not None:
            at_lawyers_extra_filters.append(date_filter)
            at_lawyers_final_filters = " AND ".join(at_lawyers_extra_filters)
        else:
            at_lawyers_final_filters = date_filter

        at_lawyers_selected_fields = [
            "LAWY_ID",
        ]

        at_lawyers = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_lawyers",
            at_lawyers_final_filters,
            at_lawyers_selected_fields,
        )

        # ============================== dim_packet ====================
        dim_packet_extra_filters = None
        if dim_packet_extra_filters is not None:
            dim_packet_extra_filters.append(date_filter)
            dim_packet_final_filters = " AND ".join(dim_packet_extra_filters)
        else:
            dim_packet_final_filters = date_filter

        dim_packet_selected_fields = [
            "PACK_ID",
            "PACK_NAME",
        ]

        dim_packet = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_packet",
            dim_packet_final_filters,
            dim_packet_selected_fields,
        )

        # ================== dim_customer_static ==================

        dim_customer_static_extra_filters = None

        if dim_customer_static_extra_filters is not None:
            dim_customer_static_extra_filters.append(date_filter)
            dim_customer_static_final_filters = " AND ".join(
                dim_customer_static_extra_filters
            )
        else:
            dim_customer_static_final_filters = date_filter

        dim_customer_static_selected_fields = [
            "CUST_CIF",
            "CUST_ID",
        ]

        dim_customer_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_customer_static",
            dim_customer_static_final_filters,
            dim_customer_static_selected_fields,
        )

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "portfolio_id_qca",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # =================== Join Conditions ==============================

        LegalExpensesExt_FsCasesSnap_join_condition = (
            fs_legal_expenses_ext["CASE_CODE"] == fs_cases_snap["CASE_CODE"]
        )

        FsCasesSnap_DimCasesStatic_join_condition = (
            fs_cases_snap["CASE_CODE"] == dim_cases_static["CASE_CODE"]
        )

        LegalExpensesExt_FsActivityLeaf_join_condition = (
            fs_legal_expenses_ext["CAC_CODE"] == fs_activity_leaf["CAC_CODE"]
        )

        LegalExpensesExt_DimLegalExpenseType_join_condition = (
            fs_legal_expenses_ext["LGET_ID"] == dim_legal_expense_type["LGET_ID"]
        )

        LegalExpensesExt_AtLawyers_join_condition = (
            fs_legal_expenses_ext["LAWY_ID"] == at_lawyers["LAWY_ID"]
        )

        DimPacket_FsActivityLeaf_join_condition = (
            dim_packet["PACK_ID"] == fs_activity_leaf["PACK_ID"]
        )

        dim_customer_static_fs_cases_snap_join_condition = (
            dim_customer_static["CUST_ID"] == fs_cases_snap["CUST_CODE"]
        )

        fs_cases_snap_portfolio_join_condition = (
            fs_cases_snap["SPET_CODE"] == portfolio["portfolio_id_qca"]
        )

        # ========================= Apply the Joins ================

        Joined_df = (
            fs_legal_expenses_ext.join(
                fs_cases_snap,
                LegalExpensesExt_FsCasesSnap_join_condition,
                "inner",
            )
            .join(
                dim_cases_static,
                FsCasesSnap_DimCasesStatic_join_condition,
                "inner",
            )
            .join(
                fs_activity_leaf,
                LegalExpensesExt_FsActivityLeaf_join_condition,
                "left",
            )
            .join(
                dim_legal_expense_type,
                LegalExpensesExt_DimLegalExpenseType_join_condition,
                "left",
            )
            .join(
                at_lawyers,
                LegalExpensesExt_AtLawyers_join_condition,
                "left",
            )
            .join(
                dim_packet,
                DimPacket_FsActivityLeaf_join_condition,
                "left",
            )
            .join(
                dim_customer_static,
                dim_customer_static_fs_cases_snap_join_condition,
                "left",
            )
            .join(
                portfolio,
                fs_cases_snap_portfolio_join_condition,
                "left",
            )
        )

        # ================= Set the Keys for Hashkey================

        ExpenseId_primary_key_fields = [
            fs_legal_expenses_ext.LEED_CODE,
            lit("qca"),
        ]

        # =============== Creation of Hashkey ===================

        target_table_qca = Joined_df.withColumn(
            "ExpensesId",
            sha2(
                concat_ws(";", *ExpenseId_primary_key_fields),
                256,
            ),
        )

        # =========== Selection of Final Columns ================

        result_df_qca = target_table_qca.select(
            target_table_qca.ExpensesId.cast("string").alias("ExpensesId"),
            lit("qca").cast("string").alias("SourceSystem"),
            portfolio.portfolio_id_qca.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            fs_legal_expenses_ext.LEED_CODE.cast("string").alias("ExpensesKey"),
            fs_cases_snap.CUST_CODE.cast("string").alias("PartyKey"),
            dim_customer_static.CUST_CIF.cast("string").alias("CDI"),
            dim_cases_static.CASE_ACCOUNT_FOR_SEARCH.cast("string").alias(
                "ExposureKey"
            ),
            dim_cases_static.CASE_CONTRACT_ACCOUNT_KEY.cast("string").alias(
                "AgreementKey"
            ),
            fs_legal_expenses_ext.CAC_CODE.cast("string").alias("LegalActionKey"),
            lit(None).cast("string").alias("InsuranceKey"),
            fs_legal_expenses_ext.LGET_ID.cast("string").alias("ExpensesTypeCode"),
            dim_legal_expense_type.LGET_NAME.cast("string").alias("ExpensesType"),
            fs_legal_expenses_ext.PD_DESC.cast("timestamp").alias("ExpensesDate"),
            fs_legal_expenses_ext.FSLD_TOTAL_AMT_LCY.cast("decimal(20,6)").alias(
                "TotalAmount"
            ),
            fs_legal_expenses_ext.FSLD_NET_AMT_LCY.cast("decimal(20,6)").alias(
                "NetAmount"
            ),
            fs_legal_expenses_ext.FSLD_TAX_AMT_LCY.cast("decimal(19,4)").alias(
                "TaxAmount"
            ),
            fs_legal_expenses_ext.FSLD_VAT_AMT_LCY.cast("decimal(20,6)").alias(
                "VatAmount"
            ),
            fs_activity_leaf.FSAC_INTERNAL_COST_LCY.cast("decimal(19,10)").alias(
                "InternalCost"
            ),
            fs_activity_leaf.FSAC_OTHER_EXPENSES_LCY.cast("decimal(19,10)").alias(
                "OtherExpenses"
            ),
            lit(None).cast("string").alias("ExpensesCategoryCode"),
            lit(None).cast("string").alias("ExpensesCategory"),
            lit(None).cast("string").alias("ExpensesStatusCode"),
            lit(None).cast("string").alias("ExpensesStatus"),
            lit(None).cast("string").alias("VATPercentage"),
            fs_activity_leaf.PACK_ID.cast("int").alias("PacketCode"),
            dim_packet.PACK_NAME.cast("string").alias("Packet"),
            lit(None).cast("string").alias("Comments"),
            at_lawyers.LAWY_ID.cast("int").alias("LawyerKey"),
            fs_legal_expenses_ext.EXTC_ID.cast("int").alias("ExternalCompanyPartyKey"),
            fs_legal_expenses_ext.LEE_INVOICE_DATE.cast("timestamp").alias(
                "InvoiceDate"
            ),
            fs_legal_expenses_ext.LEE_INVOICE_NUMBER.cast("string").alias(
                "InvoiceNumber"
            ),
            fs_legal_expenses_ext.LEEA_ID.cast("int").alias(
                "InvoiceApprovalStatusCode"
            ),
            fs_legal_expenses_ext.LEEA_ID.cast("int").alias("InvoiceApprovalStatus"),
            fs_legal_expenses_ext.LEEDV_ID.cast("int").alias(
                "ExpensesValidationStatusCode"
            ),
            fs_legal_expenses_ext.LEEDV_ID.cast("int").alias(
                "ExpensesValidationStatus"
            ),
            fs_legal_expenses_ext.LEEDA_ID.cast("int").alias(
                "ExpensesApprovalStatusCode"
            ),
            fs_legal_expenses_ext.LEEDA_ID.cast("int").alias("ExpensesApprovalStatus"),
            fs_legal_expenses_ext.Businessdate.cast("date").alias("BusinessDate"),
        )

        # ================== Append ==================

        result_df_qca.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_expenses_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)